<a href="https://colab.research.google.com/github/Stamatics-NumberstoNeurons/assignment-3-YuvasankarSelvan/blob/main/mideval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from re import X
import numpy as np
import math

class Value :
  def __init__(self,data, _children=(), _op=''):
    self.data = data
    self.grad = 0.0
    self.op = _op
    self._prev = set(_children)
    self._backward = lambda : None

  def __add__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += out.grad
      other.grad += out.grad

    out._backward = _backward
    return out


  def __sub__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data - other.data, (self, other), '-')

    def _backward():
      self.grad += out.grad
      other.grad += out.grad

    out._backward = _backward
    return out

  def __mul__(self,other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other),'*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def sigmoid(self):
    return Value(1 /(1 - np.exp(self.data)))

  def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

  def relu(self):
    out = Value( max(0.0, self.data), (self,), 'relu')
    def _backward():
      self.grad += (self.data > 0) * out.grad
    out._backward = _backward
    return out

  def __repr__(self):
    return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

  def __radd__(self,other): return self + other
  def __rmul__(self,other): return self * other
  def __rsub__(self,other): return self - other

x1,x2 = 2.0, 3.0

w1 = Value(np.random.uniform(0,1))
w2 = Value(np.random.uniform(0,1))

target = 1.0
lr = 0.1
losses = []

for step in range(100):
  # a1 = relu(x1w11 + x2w21)  a2 = relu(w12a1)

  #forward pass:
  z = w1*x1 + w2*x2
  pred = z.sigmoid()

  #loss
  loss = Value(0.5) * (pred + Value(-1)*target) * (pred + Value(-1)*target)
  losses.append(loss.data)

  # manual gradient
  dpred = pred - target
  dz = pred*(1-pred)
  dw1 = dz*x1
  dw2 = dz*x2

  # update:
  w1 -= lr * dw1
  w2 -= lr * dw2

print('final loss:',losses[-1])
print("final pred:",pred.data,". target:",target)

final loss: 1.0555974944870676e-08
final pred: 1.000145299517858 . target: 1.0


In [ ]:
from re import X
import numpy as np
import math

class Value :
  def __init__(self,data, _children=(), _op=''):
    self.data = data
    self.grad = 0.0
    self.op = _op
    self._prev = set(_children)
    self._backward = lambda : None

  def __add__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += out.grad
      other.grad += out.grad

    out._backward = _backward
    return out


  def __sub__(self,other):
    other = other if isinstance(other,Value) else Value(other)
    out = Value(self.data - other.data, (self, other), '-')

    def _backward():
      self.grad += out.grad
      other.grad += out.grad

    out._backward = _backward
    return out

  def __mul__(self,other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other),'*')

    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
    return out

  def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

  def relu(self):
    out = Value( max(0.0, self.data), (self,), 'relu')
    def _backward():
      self.grad += (self.data > 0) * out.grad
    out._backward = _backward
    return out

  def __repr__(self):
    return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

  def __radd__(self,other): return self + other
  def __rmul__(self,other): return self * other

x1,x2 = 2.0, 3.0

w11 = Value(np.random.uniform(0,1))
w12 = Value(np.random.uniform(0,1))
w21 = Value(np.random.uniform(0,1))
w22 = Value(np.random.uniform(0,1))
w1 = Value(np.random.uniform(0,1))
w2 = Value(np.random.uniform(0,1))

target = 1.0
lr = 0.1
losses = []

for step in range(100):
  # a1 = relu(x1w11 + x2w21)  a2 = relu(w12a1)

  #forward pass:
  z1 = x1*w11 + x2*w12
  z2 = x1*w21 + x2*w22
  a1 = z1.relu()
  a2 = z2.relu()
  z3 = a1*w1 + a2*w2
  pred = z3.relu()

  #loss
  loss = Value(0.5) * (pred + Value(-1)*target) * (pred + Value(-1)*target)
  losses.append(loss.data)

  # manual gradient
  dpred = pred - target
  dz3 = dpred * (z3.data > 0)
  da1 = dz3 * w1
  da2 = dz3 * w2
  dw1 = dz3 * a1
  dw2 = dz3 * a2
  dz1 = da1 * (z1.data > 0)
  dz2 = da2 * (z2.data > 0)
  dw11 = dz1 * x1
  dw12 = dz1 * x2
  dw21 = dz2 * x1
  dw22 = dz2 * x2

  # update:
  w1 -= lr * dw1
  w2 -= lr * dw2
  w11 -= lr * dw11
  w12 -= lr * dw12
  w21 -= lr * dw21
  w22 -= lr * dw22

print('final loss:',losses[-1])
print("final pred:",pred.data,". target:",target)

final loss: 3.0763656325309627e-15
final pred: 0.9999999215606523 . target: 1.0


**Modifications :**
> * the activation function here is ReLU, ReLU(x) = max(0.0,x)
> * It has 1 hidden layer.

**Mathematical Changes :**
> * The Sigmoid function has a range of (0,1) and derivative of sigmoid peaks at x = 0 and approaches 0 as |x| increases.
> * On the other hand, The ReLU has a range of (0,∞) and derivative is 0 for negative values and 1 for positive values.
> * Increasing no of hidden layers increases computation and complexity of neural network which allows to train effectively and predict more accurately.

**Behavioural difference :**
> * Adding a hidden layer gives some non-linearity to the prediction process, allowing the computer to go beyond linear decision boundaries.
> * The ReLU activation function, fixes the saturation problem that was in sigmod activation function ( the gradient goes to 0 as value become more positive or more negative ).
> * But the flaw in ReLU is the situation of **dying ReLU**. Since the gradient for negative number is 0, if a neuron gets a large negative bias during training it will always output 0. if the gradient becomes 0, it will never learn or update again.